# 04 · Cleanup

Delete the cluster the previous notebooks used. Bifrost would reap it on the profile's idle
timeout or TTL anyway; this is the polite version. Set `BIFROST_CLUSTER_ID` to pick one when
several are running.

In [ ]:
# The Bifrost sidebar talks to a small server extension inside this very
# JupyterLab (`/user/<you>/bifrost/*`). A notebook can call the same routes
# with the server's own hub token, so what happens here is exactly what a
# click in the sidebar does: same identity, same project, same NetworkPolicy.
import os, time, json, requests

SERVER = os.environ["JUPYTERHUB_SERVICE_URL"]          # http://0.0.0.0:8888/user/<you>/
USER = os.environ.get("JUPYTERHUB_USER", "me")
_HDR = {"Authorization": f"token {os.environ['JUPYTERHUB_API_TOKEN']}"}

def ext(method, path, body=None, **kw):
    """Call an extension route; returns (status, json-or-text)."""
    r = requests.request(method, SERVER + "bifrost/" + path, headers=_HDR, json=body, timeout=60, **kw)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text

def my_running_cluster():
    """The cluster these notebooks share: BIFROST_CLUSTER_ID if set, else the one running cluster."""
    want = os.environ.get("BIFROST_CLUSTER_ID")
    status, view = ext("GET", "clusters")
    assert status == 200 and view.get("configured", True), f"extension not configured: {status} {view}"
    running = [c for c in view["clusters"] if c["state"] == "running"]
    if want:
        return next((c for c in running if c["id"] == want), None)
    return running[0] if len(running) == 1 else None

def wait_running(cluster_id, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        status, view = ext("GET", "clusters")
        state = next((c["state"] for c in view["clusters"] if c["id"] == cluster_id), "gone")
        print(f"{time.time()-t0:5.0f}s  {cluster_id}: {state}", flush=True)
        if state == "running":
            return
        if state in ("failed", "terminated", "gone"):
            raise RuntimeError(f"cluster {cluster_id} went {state}")
        time.sleep(10)
    raise TimeoutError(f"cluster {cluster_id} not running after {timeout}s")

print("notebook user:", USER, "| server:", SERVER)

In [ ]:
cluster = my_running_cluster()
if cluster is None:
    print("nothing running to delete")
else:
    status, r = ext("DELETE", f"clusters/{cluster['id']}")
    print(status, r)
    assert status == 200, r
    t0 = time.time()
    while time.time() - t0 < 300:
        status, view = ext("GET", "clusters")
        if all(c["id"] != cluster["id"] or c["state"] in ("terminated", "terminating") for c in view["clusters"]):
            print(f"gone after {time.time()-t0:.0f}s"); break
        time.sleep(5)